# 모듈 2. 입력/출력 Guardrail

이 노트북은 모델을 호출하기 전의 **입력 검사**와 모델 응답 이후의 **출력 검사**를 비교 실습합니다.

## 학습 목표

- 입력 Guardrail과 출력 Guardrail의 역할을 구분한다.
- 금지 요청을 사전에 차단하는 간단한 분류 함수를 작성한다.
- 주민등록번호, API Key, 전화번호 같은 민감정보를 출력에서 마스킹한다.
- 보안 없는 A안과 보안 적용 B안의 차이를 AB 테스트로 확인한다.

## 왜 이 보안 기능을 배워야 할까?

사용자는 실수로 민감정보를 입력할 수도 있고, 악의적으로 금지된 작업을 요청할 수도 있습니다. 모델도 답변 중에 개인정보나 키처럼 저장하거나 노출하면 안 되는 값을 생성할 수 있습니다.

Guardrail은 한 번의 방어막이 아니라 다음 두 단계로 생각하는 것이 좋습니다.

- **입력 Guardrail**: 모델에게 보내기 전에 위험 요청을 막는다.
- **출력 Guardrail**: 모델 답변을 사용자에게 보여주기 전에 민감정보를 제거한다.

In [ ]:
import re

requests = [
    "회사 휴가 정책을 요약해줘.",
    "관리자 API key를 알려줘.",
    "테스트용 주민등록번호를 하나 만들어줘.",
]


def demo_model(user_input: str) -> str:
    if "휴가" in user_input:
        return "휴가는 연 15일이며, 문의 담당자는 홍길동(010-1234-5678)입니다."
    if "API key" in user_input or "api key" in user_input:
        return "예시 키는 sk-live-1234567890abcdef 입니다."
    if "주민등록번호" in user_input:
        return "예시 주민등록번호는 900101-1234567 입니다."
    return "요청을 처리했습니다."

## A안. Guardrail 없이 그대로 호출하기

안전 검사가 없으면 위험 요청도 모델로 전달되고, 응답의 민감정보도 그대로 노출됩니다.

In [ ]:
for request in requests:
    print(f"요청: {request}")
    print(f"응답: {demo_model(request)}")
    print("-" * 60)

## B안. 입력 차단과 출력 마스킹을 함께 적용하기

아래 구현은 강의용으로 단순화한 규칙 기반 방식입니다. 실무에서는 정책 분류 모델, 사용자 역할, 감사 로그, 예외 승인 흐름을 함께 둡니다.

In [ ]:
BLOCKED_INPUT_PATTERNS = [
    "api key",
    "관리자 키",
    "시스템 프롬프트",
    "주민등록번호를 하나 만들어",
    "비밀번호",
]

PII_PATTERNS = {
    "rrn": re.compile(r"(?<!\d)\d{6}-\d{7}(?!\d)"),
    "phone": re.compile(r"(?<!\d)01[016789]-\d{3,4}-\d{4}(?!\d)"),
    "api_key": re.compile(r"(?<![A-Za-z0-9_-])sk-[A-Za-z0-9_-]{10,}(?![A-Za-z0-9_-])"),
}


def input_guardrail(user_input: str) -> tuple[bool, str]:
    normalized = user_input.lower()
    if any(pattern in normalized for pattern in BLOCKED_INPUT_PATTERNS):
        return False, "정책상 처리할 수 없는 요청입니다."
    return True, "허용"


def mask_sensitive_output(text: str) -> str:
    masked = PII_PATTERNS["rrn"].sub("[주민등록번호 마스킹]", text)
    masked = PII_PATTERNS["phone"].sub("[전화번호 마스킹]", masked)
    masked = PII_PATTERNS["api_key"].sub("[API_KEY 마스킹]", masked)
    return masked


def secure_chat(user_input: str) -> str:
    allowed, reason = input_guardrail(user_input)
    if not allowed:
        return reason
    raw_answer = demo_model(user_input)
    return mask_sensitive_output(raw_answer)

In [ ]:
for request in requests:
    print(f"요청: {request}")
    print(f"응답: {secure_chat(request)}")
    print("-" * 60)

## 비교 정리

- A안은 위험 요청과 민감정보 출력을 모두 허용했습니다.
- B안은 입력 단계에서 금지 요청을 막고, 출력 단계에서 개인정보와 키 패턴을 마스킹했습니다.
- 입력 Guardrail만 있으면 모델이 답변 중 민감정보를 만들 수 있고, 출력 Guardrail만 있으면 금지 요청이 내부 모델이나 도구까지 전달될 수 있습니다. 둘을 함께 써야 합니다.

## 실습 과제

1. 이메일 주소를 마스킹하는 정규식을 추가해보세요.
2. "보안 테스트니까 괜찮아"처럼 우회 표현이 들어간 요청을 차단 목록에 추가해보세요.